In [3]:
# ==============================================================================
# SILVER LAYER TRANSFORMATION NOTEBOOK
# Purpose: Currency Normalization, UTC Conversion, PII Masking & Delta Parquet Write
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_utc_timestamp, sha2, current_timestamp, when
from delta.tables import DeltaTable

# 1. Read Raw JSON data from Bronze Layer OneLake Shortcut
bronze_path = "abfss://Ecommerce_Clickstream_Analytics@onelake.dfs.fabric.microsoft.com/lh_ecommerce_bronze.Lakehouse/Files/raw-clickstream/*.json"
df_raw = spark.read.json(bronze_path)

print(f"Total raw records read from Bronze Layer: {df_raw.count()}")
display(df_raw.limit(5))

# 2. Define Currency Exchange Rates to USD
# (In production, this can be dynamically fetched from an exchange rate API table)
df_normalized = df_raw.withColumn(
    "usd_price",
    col("local_price") / 
    when(col("local_currency") == "INR", 83.0)
    .when(col("local_currency") == "GBP", 0.79)
    .when(col("local_currency") == "EUR", 0.92)
    .otherwise(1.0) # Default for USD
)

# 3. Timestamp Normalization to UTC & Drop Duplicates
df_cleaned = df_normalized \
    .withColumn("event_timestamp_utc", to_utc_timestamp(col("event_timestamp"), "UTC")) \
    .dropDuplicates(["event_id"])

# 4. PII Data Masking (GDPR Compliance using SHA-256 Hashing for Email & IP)
df_secured = df_cleaned \
    .withColumn("masked_user_email", sha2(col("user_email"), 256)) \
    .withColumn("masked_ip_address", sha2(col("ip_address"), 256)) \
    .drop("user_email", "ip_address", "local_price", "local_currency") # Drop raw PII and raw local price

# Add processing metadata column
df_silver = df_secured.withColumn("processed_at_utc", current_timestamp())

print("Transformed Silver DataFrame Schema:")
df_silver.printSchema()
display(df_silver.limit(5))

# ==============================================================================
# 5. Write to Silver Layer as Delta Parquet Table
# ==============================================================================
silver_table_name = "silver_global_clicks"

target_path = "abfss://Ecommerce_Clickstream_Analytics@onelake.dfs.fabric.microsoft.com/lh_ecommerce_bronze.Lakehouse/Tables/silver_global_clicks"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(target_path)

print(f"Successfully written transformed data to Silver Delta Path: {target_path}")
# ==============================================================================
# 6. Performance Optimization (Z-Ordering on frequently queried columns)
# ==============================================================================
print("Optimizing Delta Table with Z-Ordering...")
target_path = "abfss://Ecommerce_Clickstream_Analytics@onelake.dfs.fabric.microsoft.com/lh_ecommerce_bronze.Lakehouse/Tables/silver_global_clicks"
spark.sql(f"OPTIMIZE delta.`{target_path}` ZORDER BY (user_id, country_code)")

print("Delta Table Optimization & Z-Ordering Completed Successfully!")

StatementMeta(, b7a68de4-761f-4f8e-86e3-a5f8ff3c2368, 5, Finished, Available, Finished, False)

Total raw records read from Bronze Layer: 110


SynapseWidget(Synapse.DataFrame, 9ba4da37-52bf-428b-80bd-f9825dffa2b5)

Transformed Silver DataFrame Schema:
root
 |-- country_code: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- usd_price: double (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- masked_user_email: string (nullable = true)
 |-- masked_ip_address: string (nullable = true)
 |-- processed_at_utc: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, b5f24d11-639c-42ba-8232-cfd117bab5f5)

Successfully written transformed data to Silver Delta Path: abfss://Ecommerce_Clickstream_Analytics@onelake.dfs.fabric.microsoft.com/lh_ecommerce_bronze.Lakehouse/Tables/silver_global_clicks
Optimizing Delta Table with Z-Ordering...
Delta Table Optimization & Z-Ordering Completed Successfully!
